# Cycle 4 — APIs and Reproducible Acquisition

**Essential question:** How can we request live information responsibly while preserving a result that another person can reproduce?

In [1]:
%pip install requests

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. API and HTTP Model

An API is a documented interface. A GET request combines a base endpoint, parameters, and headers. A response has a status code, headers, and body. JSON represents objects as dictionaries and arrays as lists.

In [2]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

DATA_DIR = Path("../artofdata-04-api-intro")
FIXTURE_DIR = DATA_DIR / "api-fixtures"
assert FIXTURE_DIR.exists(), "Run from the teaching-notebook directory and keep datasets beside it."

In [3]:
fixture_path = FIXTURE_DIR / "charizard.json"
with fixture_path.open(encoding="utf-8") as handle:
    fixture = json.load(handle)
fixture_path_fire = FIXTURE_DIR / "fire.json"
with fixture_path_fire.open(encoding="utf-8") as handle:
    type_fixture = json.load(handle)
print(fixture.keys())

print(fixture["name"]) # route param is id or name, and it returns a basic profile of the pokemon
print(fixture["abilities"]) # route parameter with the abilities of charizard

dict_keys(['id', 'name', 'base_experience', 'height', 'is_default', 'order', 'weight', 'abilities', 'past_abilities', 'forms', 'game_indices', 'held_items', 'location_area_encounters', 'moves', 'species', 'sprites', 'cries', 'stats', 'past_stats', 'types', 'past_types'])
charizard
[{'is_hidden': False, 'slot': 1, 'ability': {'name': 'blaze', 'url': 'https://pokeapi.co/api/v2/ability/66/'}}, {'is_hidden': True, 'slot': 3, 'ability': {'name': 'solar-power', 'url': 'https://pokeapi.co/api/v2/ability/94/'}}]


### Main Keys in the Charizard response
- `id`: Pokemon's number on the pokedex, charizard's being 6
- `name`: name of the Pokemon
- `height` / `weight`: physical measurements of the Pokemon
- `base_experience`: EXP from defeating Pokemon
- `abilities`: list of abilities, including hidden
- `types`: types: type of pokemon, like flying or fire
- `stats`: six stats

## 2. Read Documentation Before Requesting

For PokeAPI's `/pokemon/{id or name}`, the route parameter is the Pokémon's id or name (I used `charizard` because he's the only one I know). No API key needed. `limit` and `offset` are query parameters, but they only apply to list endpoints like `/pokemon/`, not to a single Pokémon, and their uses are commented. I also used `/type/{id or name}` with `fire` to get type matchups.

In [4]:
ENDPOINT = "https://pokeapi.co/api/v2/pokemon/charizard"

# Limit: controls how many items received during a single request
# Offset: controls where to start counting from the master list
# ENDPOINTS: ability/name, type/id, move/id
PARAMS = {
    "limit": 20,
    "offset": 0,
}

EXPECTED_KEYS = {"id", "name", "height", "weight", "abilities", "types", "stats"}

## 3. Validations

Before using the data, I check that it has everything I expect. The first two functions check that each payload is a dictionary, as the example given in the tutorial notebook does. Then I check `damage_relations` for all 6 damage keys, and `stats` for exactly 6 . I run the checks on the fixtures first then on live data.

In [5]:
POKEMON_KEYS = {"id", "name", "height", "weight", "abilities", "types", "stats"}
TYPE_KEYS = {"id", "name", "damage_relations", "pokemon", "moves"}
import requests
def json1(url):
    # timeout=10: if the server doesn't respond within 10 seconds, give up
    response = requests.get(url, timeout=10)
    # raise_for_status() raises an HTTPError for any 404 or 403, or 5xx code, like page not found or internal bug
    response.raise_for_status()
    return response.json() 


def validate_pokemon_payload(payload):
    if not isinstance(payload, dict):  # is the payload a dict
        raise TypeError("API payload must be a JSON object")
    missing = POKEMON_KEYS - set(payload)  #expected keys are missing
    if missing:
        raise ValueError(f"Missing fields: {sorted(missing)}")
    return payload
# same functions basically as example

def validate_type_payload(payload):
    if not isinstance(payload, dict): 
        raise TypeError("API payload must be a JSON object")
    missing = TYPE_KEYS - set(payload)  
    if missing:
        raise ValueError(f"Missing fields: {sorted(missing)}")
    return payload

print(type_fixture["damage_relations"].keys()) #find how many keys of damage relation
DAMAGE_RELATION_KEYS = {
    "double_damage_from", "double_damage_to", "half_damage_from", "half_damage_to", "no_damage_from", "no_damage_to",
}
#Similar check on damage_relation keys in fire type
def validate_damage_relations(payload):
    relations = payload["damage_relations"]
    missing = DAMAGE_RELATION_KEYS - set(relations)
    if missing:
        raise ValueError(f"Missing keys: {sorted(missing)}")
# check for pokemon stats, and since pokemon has 6 stats, if there is missing one we must flag
def validate_stats(payload):
    if len(payload["stats"]) != 6:
        raise ValueError("6 stats was expected...")
#validate using top functions first
validate_pokemon_payload(fixture)
validate_type_payload(type_fixture)

#then specialized validations
validate_stats(fixture)
validate_damage_relations(type_fixture)

live_poke = json1(ENDPOINT) # live just like down below but simpler
live_type = json1("https://pokeapi.co/api/v2/type/fire") # live just like down below
validate_stats(live_poke)
validate_damage_relations(live_type)
print("Yay") # if all goes well

dict_keys(['no_damage_to', 'half_damage_to', 'double_damage_to', 'no_damage_from', 'half_damage_from', 'double_damage_from'])
Yay


### Live/fixture validation
This function defaults to the local fixture, in case there is something wrong with live. When use_live is True, it requests the live endpoint with a 10-second timeout, checks the HTTP statu for codes like 400 and the teapot, parses the JSON, and validates it. If any step fails, it prints why and falls back to the fixture, so the notebook always runs. `acquisition_source` tells me whether the data came from the live URL or the cache.

In [6]:
def acquire_pokemon(use_live=False):
    source = "cached fixture"
    payload = fixture #setting payload = fixture
    if use_live: # if this is false, than it uses the fixture
        try:
            response = requests.get(ENDPOINT, timeout=10) # get request using requests library, route+query params go into endpoint, the rest is unnecessary without header 
            response.raise_for_status() # status codes
            payload = response.json() 
            validate_pokemon_payload(payload) 
            source = response.url
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = fixture
    return validate_pokemon_payload(payload), source

pokemon_payload, acquisition_source = acquire_pokemon(use_live=True)
acquisition_source

'https://pokeapi.co/api/v2/pokemon/charizard'

## 5. Normalize and Record Provenance

The `stats` list is nested, each stat's name is inside another dictionary. When I saw the output using only pd.DataFrame to the stats, the stats column, `{name: hp and url: https://pokeapi.co/api... `, looked very awkward. I used this new function,`pd.json_normalize` which takes each key in the nested dictionary and pulls it out to a form a new column, and the values for the key makes the row values. Now after I changed pd.DataFrame to pd.json_normalize, the name and url columns look much better. The default naming rule of json_normalize is to take the path of the nested dictionaries, so it would be stat.name and stat.url for column names by default, but for even better readability, I changed them.

The asserts confirm there are 6 rows and the columns I need.

json_normalize: https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html 

In [7]:
stats = pd.json_normalize(pokemon_payload["stats"])
stats = stats.rename(columns={"stat.name": "stat", "stat.url": "url"})
assert len(stats) == 6
assert {"stat", "base_stat", "effort"} <= set(stats.columns)
stats

,base_stat,effort,stat,url
0,78,0,hp,https://pokeapi.co/api/v2/stat/1/
1,84,0,attack,https://pokeapi.co/api/v2/stat/2/
2,78,0,defense,https://pokeapi.co/api/v2/stat/3/
3,109,3,special-attack,https://pokeapi.co/api/v2/stat/4/
4,85,0,special-defense,https://pokeapi.co/api/v2/stat/5/
5,100,0,speed,https://pokeapi.co/api/v2/stat/6/


In [8]:
provenance = { # https://github.com/PokeAPI/pokeapi/blob/master/LICENSE.md - License information
    "provider": "PokeAPI",
    "endpoint": ENDPOINT,
    "parameters": PARAMS,
    "acquisition_source": acquisition_source,
    "fixture_retrieved": "2026-11-13",
    "license": "BSD 3-Clause, Copyright Paul Hallett and PokeAPI contributors. Pokémon names are trademarks of Nintendo.",
}
provenance 

{'provider': 'PokeAPI',
 'endpoint': 'https://pokeapi.co/api/v2/pokemon/charizard',
 'parameters': {'limit': 20, 'offset': 0},
 'acquisition_source': 'https://pokeapi.co/api/v2/pokemon/charizard',
 'fixture_retrieved': '2026-11-13',
 'license': 'BSD 3-Clause, Copyright Paul Hallett and PokeAPI contributors. Pokémon names are trademarks of Nintendo.'}

### DataFrame Checks

In [9]:
print(stats.shape)                               # expect (6, 4), 6 for 6 stats, 4 for columns
print(stats.dtypes)                              # base_stat and effort should be integers
print(stats.isna().sum())                        # expect zeroes
print(stats["stat"].duplicated().sum())          # expect 0 duplicates
print(stats["base_stat"].between(1, 255).all())  # expecting True, I looked up the plausible ranges between pokemon stats, although most of them fall between 90 and 120

(6, 4)
base_stat    int64
effort       int64
stat           str
url            str
dtype: object
base_stat    0
effort       0
stat         0
url          0
dtype: int64
0
True


The table has 6 rows and 4 columns. `base_stat` and `effort` are integers. There are no missing values and no duplicate stats, and every base stat is between 1 and 255, the plausible range. The data is clean. 